# Metasyn multiple table tutorial

In this tutorial you will learn how to create synthetic versions of multiple tables at onc, while preserving some relations between tables.

First you should install metasyn if you have not done so already.

In [1]:
# %pip install metasyn

### Loading the dataset

We will use a demonstration dataset that is built into metasyn, called `ShopMultiDataset`. This dataset contains three tables that are interrelated through customer id's and product id's.



In [2]:
from metasyn.demo.dataset import ShopMultiDataset
from metasyn.multiframe import MultiFrame

Here we load in our demo dataset. In your own usecase, you simply need to read the files into
a dictionary of data frames.
For example: ``data = {"name_1": pl.read_csv("file1.csv"), ...}``

In [3]:
data = ShopMultiDataset().get_dataframes()
print({key: d.head(1) for key, d in data.items()})

{'customers': shape: (1, 4)
┌───────┬──────────────────────────────┬─────────────────┬─────────────┐
│ id    ┆ address                      ┆ credit_card_nr  ┆ signup_date │
│ ---   ┆ ---                          ┆ ---             ┆ ---         │
│ i64   ┆ str                          ┆ i64             ┆ date        │
╞═══════╪══════════════════════════════╪═════════════════╪═════════════╡
│ 21330 ┆ 55036 Buchanan Loaf Apt. 324 ┆ 372318430068714 ┆ 1978-04-15  │
│       ┆ S…                           ┆                 ┆             │
└───────┴──────────────────────────────┴─────────────────┴─────────────┘, 'products': shape: (1, 4)
┌──────┬─────────┬───────────────┬───────┐
│ id   ┆ name    ┆ current_price ┆ stock │
│ ---  ┆ ---     ┆ ---           ┆ ---   │
│ i64  ┆ str     ┆ str           ┆ i64   │
╞══════╪═════════╪═══════════════╪═══════╡
│ 1580 ┆ clearly ┆ $609.13       ┆ 3     │
└──────┴─────────┴───────────────┴───────┘, 'purchases': shape: (1, 4)
┌─────┬─────────────┬───────────

The tables have some relations between them. For example in the 'purchases' table we have the 'customer_id' column which has identifiers on who purchased that particular item. This 'customer_id' is also present in the 'customers' table. In a relational database, this is called a primary <-> foreign key relationship. This can be very useful when data from different tables have to be combined. Metasyn includes a multitable feature to capture these kinds of relations.

To examplify this, let's perform a simple join to combine the purchases and customers table:

In [4]:
data["purchases"].join(data["customers"], left_on="customer_id", right_on="id")

id,customer_id,price_paid,product_id,address,credit_card_nr,signup_date
i64,i64,str,i64,str,i64,date
0,8579,"""$78,287.51""",37931,"""68997 Kevin Summit Lake Veroni…",4103619762598105,1980-01-02
1,54694,"""$901.94""",30783,"""7809 Shepherd Orchard South Sa…",503878214520,1979-02-05
2,66228,"""$48.93""",80060,"""20642 Andrew Springs Changfurt…",3588217506423627,2014-08-02
3,53735,"""$62,333.64""",68273,"""54914 Alexis Village Pamelache…",4571100306550,2016-06-02
4,27958,"""$2.95""",3520,"""2331 Bradley Cliffs Apt. 998 B…",4979685382112,2023-07-30
…,…,…,…,…,…,…
995,48110,"""$41,246.73""",23588,"""69021 Nelson Spur East John, C…",4357547109952961,1981-04-23
996,57809,"""$3,998.16""",12079,"""0062 Ayers View Suite 421 Reed…",4239534911249873640,1994-08-03
997,50554,"""$143.18""",30135,"""2200 Solis Mountains Apt. 469 …",4929587072093553986,2000-09-02


### Synthesizing unrelated tables

Now, let us naively generate synthetic data independently using metasyn without specifying any relations.

In [5]:
multiframe = MultiFrame.fit_dataframes(data, relations=[])
syn_data = multiframe.synthesize()
# Try to join the same tables
syn_data["purchases"].join(syn_data["customers"], left_on="customer_id", right_on="id")

  product_id: 100%|██████████| 4/4 [00:00<00:00,  7.09variables/s]


id,customer_id,price_paid,product_id,address,credit_card_nr,signup_date
i64,i64,str,i64,str,i64,date


Note above that while the synthetic data has the same number of rows for the tables, the number of rows in the joined table is vastly different. This is because of the fact that the customer identifiers in the two tables are created independently.

### Synthesizing related tables
To remedy this, we can specify relations in the dataset.

In [6]:
relations = [
    "purchases[customer_id] SUBSET OF customers[id]",
    "purchases[product_id] SUBSET OF products[id]",
]

multiframe_improved = MultiFrame.fit_dataframes(data, relations=relations)
syn_data_improved = multiframe_improved.synthesize()
syn_data_improved["purchases"].join(syn_data_improved["customers"], left_on="customer_id", right_on="id")

  product_id: 100%|██████████| 4/4 [00:00<00:00,  8.23variables/s]


id,customer_id,price_paid,product_id,address,credit_card_nr,signup_date
i64,i64,str,i64,str,i64,date
0,56633,"""$8.39""",16180,"""Myself.""",3063410045022109640,1981-03-13
1,26638,"""$8.49""",51487,"""Some miss television. Ok hour …",1293916467670100581,1978-01-09
2,33109,"""$24,052.58""",9851,"""Thank.""",3698793678575972536,1976-01-11
3,33108,"""$310.00""",71339,"""Cultural speak simple.""",2456245092105149137,2005-10-03
4,18098,"""$53,934.07""",31004,"""Camera moment likely career.""",2045288769061647204,1988-08-05
…,…,…,…,…,…,…
995,26428,"""$0,178.56""",31004,"""Always level benefit son then.""",4638762806391152980,1992-08-31
996,61582,"""$496,310.37""",70844,"""Behavior debate serious new we…",2814013823906968939,1995-09-11
997,41090,"""$5,767.78""",37768,"""Population institution electio…",2144580605952112337,1970-04-16


In the presented table we only have SUBSET OF relations, but there are a few more:

- `SUBSET OF`: Column a has values that are present in column b and can occur multiple times in column a.
- `EQUALS`: Column a has the same values as column b, but not necessarily in the same order. This also implies that the table of column a and the table of column b have the same number of rows.
- `EQUAL ORDERED`: Column a has the same values as column b and also in the same order. Also implies the tables have the same number of rows.
- `INFER FROM`: The relation between column a and b should be inferred by metasyn. This will result in one of the above relationships.

We can also adjust the size of the output tables for each individual table:

In [7]:
multiframe_improved.synthesize(n={"customers": 8})

  product_id: 100%|██████████| 4/4 [00:00<00:00,  8.00variables/s]


{'customers': shape: (8, 4)
 ┌───────┬─────────────────────────────────┬─────────────────────┬─────────────┐
 │ id    ┆ address                         ┆ credit_card_nr      ┆ signup_date │
 │ ---   ┆ ---                             ┆ ---                 ┆ ---         │
 │ i64   ┆ str                             ┆ i64                 ┆ date        │
 ╞═══════╪═════════════════════════════════╪═════════════════════╪═════════════╡
 │ 29629 ┆ Administration special throw c… ┆ 1481554008638341506 ┆ 1989-02-10  │
 │ 59997 ┆ Nearly my debate here.          ┆ 959730128663445303  ┆ 2018-03-13  │
 │ 22318 ┆ Site reach everybody. Apply ra… ┆ 1912931111910970411 ┆ 2001-10-28  │
 │ 70412 ┆ Really although my this four t… ┆ 1166320154871006193 ┆ 1985-10-03  │
 │ 54554 ┆ Challenge science meeting seco… ┆ 4170223632403772682 ┆ 2005-11-02  │
 │ 63113 ┆ Spend mean rock.                ┆ 2708289156331850163 ┆ 2004-01-21  │
 │ 38693 ┆ Wonder.                         ┆ 3188382574585536709 ┆ 1973-03-11  │


### Inspecting multiframes

You can inspect multiframes with a print statement.

In [8]:
print(multiframe_improved)

Table customers:
    Number of rows: 200
    Number of columns: 4
    Columns: id, address, credit_card_nr, signup_date

Table products:
    Number of rows: 500
    Number of columns: 4
    Columns: id, name, current_price, stock

Table purchases:
    Number of rows: 1000
    Number of columns: 4
    Columns: id, customer_id, price_paid, product_id

Relations between columns:
    purchases[customer_id] SUBSET OF customers[id]
    purchases[product_id] SUBSET OF products[id]



You can select and inspect metaframes (representations of the individual tables) with brackets `[]`:

In [9]:
print(multiframe_improved["purchases"])

# Rows: 1000
# Columns: 4

Column 1: "id"
- Variable Type: discrete
- Data Type: Int64
- Proportion of Missing Values: 0.0000
- Distribution:
	- Type: core.unique_key
	- Parameters:
		- lower: 0
		- consecutive: True
	

Column 2: "customer_id"
- Variable Type: discrete
- Data Type: Int64
- Proportion of Missing Values: 0.0000
- Distribution:
	- Type: core.multinoulli
	- Parameters:
		- labels: [  216   538  1634  1869  3109  4186  4220  5936  6422  6931  7065  7080
	  8198  8442  8579  8714  9142  9175  9720  9738  9775  9827 10383 11558
	 11568 12034 12154 13065 13066 13169 13182 13535 14037 14278 14846 15318
	 16097 16216 16853 17149 17518 17998 18662 18727 19211 19884 19895 20062
	 20731 21156 21330 21453 21670 22057 22363 23038 23486 24366 24901 24955
	 25571 25838 26804 27122 27521 27958 28163 28257 28375 28418 28699 28760
	 29023 29313 29315 29586 29809 30968 31278 31666 31770 32125 32862 33416
	 33520 33954 34271 34983 35106 35147 35996 36001 37428 37545 37644 37659
	 38064 3811

### Saving and loading multiframes

Similar to metaframes, multiframes can also be saved and loaded from a .json file. This .json file is a GMF (Generative Metadata Format) file that has the same structure as when the metadata of single tables are stored.

In [10]:
multiframe.save_json("test.json")
mf = multiframe.load_json("test.json")